In [15]:
import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
from torch.utils.data import ConcatDataset, Subset, DataLoader, random_split
from torchhd import functional, embeddings
from torchhd.datasets import EuropeanLanguages as Languages
import re
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from sklearn.metrics import accuracy_score, roc_auc_score

In [2]:
def set_seed(seed=123):
    import random, numpy as np
    random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(123)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cpu')

In [3]:
DIMENSIONS     = 10_000
MAX_INPUT_SIZE = 128
BATCH_SIZE     = 32
PADDING_IDX    = 0
PRINT_EVERY    = 100

ASCII_A = ord("a")
ASCII_Z = ord("z")
ASCII_SPACE = ord(" ")
NUM_TOKENS = (ASCII_Z - ASCII_A + 1) + 1 + 1  # letters + space + PAD slot

def char2int(char: str) -> int:
    a = ord(char)
    if a == ASCII_SPACE:
        return (ASCII_Z - ASCII_A + 1)
    if ASCII_A <= a <= ASCII_Z:
        return a - ASCII_A
    return (ASCII_Z - ASCII_A + 1)  # map non a–z to space

def transform(x: str) -> torch.Tensor:
    x = x.lower()
    x = re.sub(r"\s+", " ", x)
    x = x[:MAX_INPUT_SIZE]
    ids = [char2int(ch) + 1 for ch in x]  # shift by +1 so PAD is 0
    if len(ids) < MAX_INPUT_SIZE:
        ids += [PADDING_IDX] * (MAX_INPUT_SIZE - len(ids))
    return torch.tensor(ids, dtype=torch.long)

In [32]:
random_state = 0 
ID_LANG_STRINGS = [
    # Germanic
    'English', 'German', 'Dutch', 'Swedish', 'Danish',
    # Romance
    'French', 'Italian', 'Spanish', 'Portuguese', 'Romanian',
    # Slavic
    'Czech', 'Polish', 'Slovak', 'Slovenian', 'Bulgarian',
    # Baltic
    'Latvian', 'Lithuanian',
    # Hellenic
    'Greek'
]

# Data Loading
data_root = "./data"
train_ds_raw = Languages(data_root, train=True, transform=transform, download=True)
test_ds_raw  = Languages(data_root, train=False, transform=transform, download=True)

# Map Strings to Integer Labels
all_class_names = train_ds_raw.classes
name_to_idx = {name: i for i, name in enumerate(all_class_names)}

# Verify we have all ID languages
for name in ID_LANG_STRINGS:
    if name not in name_to_idx:
        raise ValueError(f"Language {name} not found in dataset: {all_class_names}")

ID_LABELS = [name_to_idx[name] for name in ID_LANG_STRINGS]

# Full dataset
ds_full = ConcatDataset([train_ds_raw, test_ds_raw])

all_targets = []
for d in ds_full.datasets:
    all_targets.extend(d.targets)
all_targets = np.array(all_targets)

MAX_SAMPLES_PER_CLASS = 2000
subsampled_indices = []

for cls_idx in range(len(all_class_names)):
    # Get all indices for this specific class
    cls_indices = np.where(all_targets == cls_idx)[0]
    np.random.shuffle(cls_indices) 
    # Take up to the limit
    subsampled_indices.extend(cls_indices[:MAX_SAMPLES_PER_CLASS])

subsample_mask = np.zeros(len(all_targets), dtype=bool)
subsample_mask[subsampled_indices] = True

# Filter ID vs OOD using the subsampled mask
id_indices = np.where(np.isin(all_targets, ID_LABELS) & subsample_mask)[0]
ood_indices = np.where(~np.isin(all_targets, ID_LABELS) & subsample_mask)[0]

ds_id_full = Subset(ds_full, id_indices)
ds_ood = Subset(ds_full, ood_indices)

# Random Split (70% Train, 20% Calib, 10% Test)
n_total = len(ds_id_full)
n_train = int(0.75 * n_total)
n_calib = int(0.225 * n_total)
n_test  = n_total - n_train - n_calib

ds_train, ds_calib, ds_test = random_split(
    ds_id_full, [n_train, n_calib, n_test],
    generator=torch.Generator().manual_seed(random_state)
)

# Loaders
ld_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True)
ld_calib = DataLoader(ds_calib, batch_size=BATCH_SIZE, shuffle=False)
ld_test  = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False)
ld_ood   = DataLoader(ds_ood, batch_size=BATCH_SIZE, shuffle=False)
print("Data loading and splitting complete.")
sys.stdout.flush()

Files already downloaded and verified
Files already downloaded and verified
Data loading and splitting complete.


In [33]:
print(f"Train size: {len(ds_train)}, calib size: {len(ds_calib)}, test size: {len(ds_test)}\n")
print(f"Train size per class: {int(len(ds_train)/len(ID_LANG_STRINGS))}, " +
      f"calib size per class: {int(len(ds_calib)/len(ID_LANG_STRINGS))}, " +
      f"test size per class: {int(len(ds_test)/len(ID_LANG_STRINGS))}\n")

Train size: 27000, calib size: 8100, test size: 900

Train size per class: 1500, calib size per class: 450, test size per class: 50



In [6]:
class Model(nn.Module):
    def __init__(self, num_classes, vocab_size, dim, padding_idx=0):
        super().__init__()
        self.symbol = embeddings.Random(vocab_size, dim, padding_idx=padding_idx)
        self.classify = nn.Linear(dim, num_classes, bias=False)
        with torch.no_grad():
            self.classify.weight.zero_()

    @torch.no_grad()
    def encode(self, x_ids: torch.Tensor) -> torch.Tensor:
        # We rely on TorchHD's ngrams (n=3) and hard_quantize, identical to the example.
        symbols = self.symbol(x_ids)                 # [B, T, D]
        hv = functional.ngrams(symbols, n=3)         # [B, D]
        hv = functional.hard_quantize(hv)            # sign -> {-1,+1}
        return hv

    def forward(self, x_ids: torch.Tensor) -> torch.Tensor:
        enc = self.encode(x_ids)                     # [B, D]
        return self.classify(enc)                    # [B, C]

model = Model(len(train_ds.classes), NUM_TOKENS, DIMENSIONS, padding_idx=PADDING_IDX).to(DEVICE)

In [6]:
import pdb
t0 = time.time()
with torch.no_grad():
    for bi, (samples, labels) in enumerate(train_ld, 1):
        samples = samples.to(DEVICE, non_blocking=True)
        labels  = labels.to(DEVICE, non_blocking=True)
        samples_hv = model.encode(samples)                          # [B, D], bipolar
        model.classify.weight.index_add_(0, labels, samples_hv)     # accumulate into class rows
        if bi % PRINT_EVERY == 0:
            print(f"[train] {bi}/{len(train_ld)}")
            print(f"  |  elapsed: {time.time() - t0:.1f}s")

    # Normalize class rows (cosine-like scoring)
    model.classify.weight[:] = F.normalize(model.classify.weight, dim=1)
    print(f"Total Time Elapsed: {time.time() - t0:.1f}s")

C:\Users\liang\AppData\Local\Temp\ipykernel_108388\1691475981.py:14: DeprecationWarning: torchhd.hard_quantize is deprecated, consider using torchhd.normalize instead.
  hv = functional.hard_quantize(hv)            # sign -> {-1,+1}


RuntimeError: [enforce fail at alloc_cpu.cpp:121] data. DefaultCPUAllocator: not enough memory: you tried to allocate 1290240000 bytes.

In [18]:
230000/21

10952.380952380952